In [ ]:
# install and load packages
install.packages(c("sqldf","RSQLite","dplyr","ggplot2","lubridate","corrplot","tidyr"),
                 repos="https://cran.r-project.org", quiet=TRUE)
library(sqldf); library(RSQLite); library(dplyr)
library(ggplot2); library(lubridate); library(corrplot); library(tidyr)

In [ ]:
# load all csvs
customers  <- read.csv("customers.csv",  stringsAsFactors=FALSE)
orders     <- read.csv("orders.csv",     stringsAsFactors=FALSE)
deliveries <- read.csv("deliveries.csv", stringsAsFactors=FALSE)
drivers    <- read.csv("drivers.csv",    stringsAsFactors=FALSE)
complaints <- read.csv("complaints.csv", stringsAsFactors=FALSE)
hubs       <- read.csv("hubs.csv",       stringsAsFactors=FALSE)
incidents  <- read.csv("incidents.csv",  stringsAsFactors=FALSE)
vehicles   <- read.csv("vehicles.csv",   stringsAsFactors=FALSE)
app_events <- read.csv("app_events.csv", stringsAsFactors=FALSE)
cat("orders:", nrow(orders), "| deliveries:", nrow(deliveries), "| customers:", nrow(customers), "\n")

In [ ]:
# clean zones - title case, fix Ctr -> Central, then parse dates
norm_zone <- function(x) {
  out <- tools::toTitleCase(tolower(trimws(x)))
  ifelse(out == "Ctr", "Central", out)
}
orders$pickup_zone  <- norm_zone(orders$pickup_zone)
orders$dropoff_zone <- norm_zone(orders$dropoff_zone)
customers$home_zone <- norm_zone(customers$home_zone)
drivers$base_zone   <- norm_zone(drivers$base_zone)
hubs$zone           <- norm_zone(hubs$zone)

deliveries$dispatch_time         <- ymd_hms(deliveries$dispatch_time)
deliveries$delivery_completed_at <- ymd_hms(deliveries$delivery_completed_at)
orders$order_created_at          <- ymd_hms(orders$order_created_at)
complaints$created_at            <- ymd_hms(complaints$created_at)
cat("Zones:", paste(sort(unique(orders$pickup_zone)), collapse=", "), "\n")

In [ ]:
# Query 1 - delivery failure rate by zone
q1 <- sqldf("SELECT o.pickup_zone,
               COUNT(*) AS total_deliveries,
               SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failed,
               ROUND(100.0*SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_rate_pct
             FROM deliveries d
             JOIN orders o ON d.order_id=o.order_id
             GROUP BY o.pickup_zone
             ORDER BY failure_rate_pct DESC")
print(q1)

In [ ]:
# Query 2 - complaint cost breakdown
q2 <- sqldf("SELECT complaint_type,
               COUNT(*) AS total,
               ROUND(AVG(resolution_days),2) AS avg_resolution_days,
               ROUND(AVG(compensation_amount),2) AS avg_compensation,
               ROUND(SUM(compensation_amount),2) AS total_compensation
             FROM complaints
             GROUP BY complaint_type
             ORDER BY total DESC")
print(q2)

In [ ]:
# Query 3 - driver performance: failure rate, overrides and rating
q3 <- sqldf("SELECT dr.driver_id, dr.base_zone, dr.training_score, dr.driver_rating,
               COUNT(*) AS total_deliveries,
               SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failures,
               ROUND(100.0*SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_pct,
               SUM(d.manual_route_override_count) AS total_overrides
             FROM drivers dr
             JOIN deliveries d ON dr.driver_id=d.driver_id
             GROUP BY dr.driver_id
             HAVING total_deliveries >= 3
             ORDER BY failure_pct DESC
             LIMIT 15")
print(q3)

In [ ]:
# Query 4 - hub performance vs failure rate and cost
q4 <- sqldf("SELECT h.hub_id, h.hub_name, h.zone, h.capacity_score,
               COUNT(*) AS total_dispatches,
               SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failures,
               ROUND(100.0*SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_pct,
               ROUND(AVG(d.fuel_or_charge_cost),2) AS avg_cost
             FROM hubs h
             JOIN deliveries d ON h.hub_id=d.hub_id
             GROUP BY h.hub_id
             ORDER BY failure_pct DESC")
print(q4)

In [ ]:
# Query 5 - do more overrides mean more failures?
options(sqldf.driver="SQLite")
q5 <- sqldf("SELECT
               CASE WHEN manual_route_override_count=0 THEN '0 overrides'
                    WHEN manual_route_override_count=1 THEN '1 override'
                    WHEN manual_route_override_count<=3 THEN '2-3 overrides'
                    ELSE '4+ overrides' END AS override_band,
               COUNT(*) AS total,
               SUM(CASE WHEN delivery_status='Failed' THEN 1 ELSE 0 END) AS failures,
               ROUND(100.0*SUM(CASE WHEN delivery_status='Failed' THEN 1 ELSE 0 END)/COUNT(*),2) AS failure_rate_pct
             FROM deliveries
             GROUP BY override_band
             ORDER BY failure_rate_pct DESC")
print(q5)

In [ ]:
# Query 6 - customers with 2+ failed deliveries and their complaint history
q6 <- sqldf("SELECT c.customer_id, c.customer_type, c.home_zone, c.loyalty_score,
               COUNT(DISTINCT o.order_id) AS total_orders,
               SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
               COUNT(DISTINCT cp.complaint_id) AS complaints
             FROM customers c
             JOIN orders o ON c.customer_id=o.customer_id
             JOIN deliveries d ON o.order_id=d.order_id
             LEFT JOIN complaints cp ON c.customer_id=cp.customer_id
             GROUP BY c.customer_id
             HAVING failed_deliveries >= 2
             ORDER BY failed_deliveries DESC
             LIMIT 15")
print(q6)

In [ ]:
# SQLite indexing benchmark - same query timed before and after index creation
con <- dbConnect(RSQLite::SQLite(), ":memory:")
dbWriteTable(con, "deliveries", deliveries)
dbWriteTable(con, "orders", orders)

bq <- "SELECT o.pickup_zone, COUNT(*) AS total,
       SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failures
       FROM deliveries d JOIN orders o ON d.order_id=o.order_id
       GROUP BY o.pickup_zone ORDER BY failures DESC"

t_before <- system.time({ dbGetQuery(con, bq) })
cat(sprintf("Without indexes: %.4f sec\n", t_before["elapsed"]))

dbExecute(con, "CREATE INDEX idx_del_order  ON deliveries(order_id)")
dbExecute(con, "CREATE INDEX idx_del_status ON deliveries(delivery_status)")
dbExecute(con, "CREATE INDEX idx_ord_zone   ON orders(pickup_zone)")

t_after <- system.time({ dbGetQuery(con, bq) })
cat(sprintf("With indexes:    %.4f sec\n", t_after["elapsed"]))

# the dataset at 950 rows is small so timing difference can be marginal,
# but EXPLAIN QUERY PLAN confirms indexes are selected over a full scan
plan <- dbGetQuery(con, paste("EXPLAIN QUERY PLAN", bq))
print(plan)

In [ ]:
# feature engineering for plots
deliveries$duration_hours <- as.numeric(
  difftime(deliveries$delivery_completed_at, deliveries$dispatch_time, units="hours"))
deliveries$duration_hours[deliveries$duration_hours < 0] <- NA
summary(deliveries$duration_hours)

In [ ]:
# Plot 1 - delivery duration histogram with mean line
ggplot(deliveries[!is.na(deliveries$duration_hours),], aes(x=duration_hours)) +
  geom_histogram(bins=40, fill="#2a9d8f", color="white", alpha=0.85) +
  geom_vline(aes(xintercept=mean(duration_hours, na.rm=TRUE)), color="red", linetype="dashed", linewidth=1) +
  labs(title="Distribution of Delivery Duration", x="Duration (Hours)", y="Count") +
  theme_minimal()

In [ ]:
# Plot 2 - failure rate per zone
zone_summary <- deliveries %>%
  left_join(orders, by="order_id") %>%
  group_by(pickup_zone) %>%
  summarise(fail_rate=round(100*mean(delivery_status=="Failed"), 2))

ggplot(zone_summary, aes(x=reorder(pickup_zone, fail_rate), y=fail_rate, fill=fail_rate)) +
  geom_col() + coord_flip() +
  scale_fill_gradient(low="#7ebf81", high="#b50000") +
  labs(title="Delivery Failure Rate by Zone", x="Zone", y="Failure Rate (%)") +
  theme_minimal()

In [ ]:
# Plot 3 - driver rating vs failure rate with regression line
driver_perf <- deliveries %>%
  left_join(drivers, by="driver_id") %>%
  group_by(driver_id, driver_rating) %>%
  summarise(fail_rate=100*mean(delivery_status=="Failed"), n=n(), .groups="drop") %>%
  filter(n >= 3)

ggplot(driver_perf, aes(x=driver_rating, y=fail_rate)) +
  geom_point(alpha=0.5, color="#0c78f2") +
  geom_smooth(method="lm", color="red", se=TRUE) +
  labs(title="Driver Rating vs Failure Rate", x="Driver Rating", y="Failure Rate (%)") +
  theme_minimal()

In [ ]:
# Plot 4 - battery health vs incident count
veh_inc <- incidents %>%
  left_join(deliveries %>% select(delivery_id, vehicle_id), by="delivery_id") %>%
  left_join(vehicles %>% select(vehicle_id, battery_health_pct), by="vehicle_id") %>%
  group_by(vehicle_id, battery_health_pct) %>%
  summarise(incident_count=n(), .groups="drop")

ggplot(veh_inc, aes(x=battery_health_pct, y=incident_count)) +
  geom_point(alpha=0.7, color="#ff7124") +
  geom_smooth(method="lm", color="black", linetype="dashed") +
  labs(title="Battery Health vs Incident Count", x="Battery Health (%)", y="Incidents") +
  theme_minimal()

In [ ]:
# Plot 5 - complaint volume over time by severity
complaints$month <- floor_date(complaints$created_at, "month")
comp_time <- complaints %>% group_by(month, severity) %>% summarise(count=n(), .groups="drop")

ggplot(comp_time, aes(x=month, y=count, color=severity, group=severity)) +
  geom_line(linewidth=1) + geom_point(size=2) +
  scale_color_manual(values=c("High"="#e63946","Medium"="#f4a261","Low"="#2a9d8f")) +
  labs(title="Complaints Over Time by Severity", x="Month", y="Count") +
  theme_minimal()

In [ ]:
# Plot 6 - correlation matrix across key delivery metrics
num_data <- deliveries %>%
  select(duration_hours, manual_route_override_count, fuel_or_charge_cost,
         customer_rating_post_delivery, delivery_status) %>%
  mutate(failed=as.numeric(delivery_status=="Failed")) %>%
  select(-delivery_status) %>% drop_na()

corrplot(cor(num_data), method="color", type="upper", addCoef.col="black",
         number.cex=0.7, tl.cex=0.8,
         title="Correlation Matrix - Delivery Metrics", mar=c(0,0,2,0))